In [1]:
import pandas as pd

In [3]:
COLUMNS = [
    "elapsed_time",
    "left_stride_interval",
    "right_stride_interval",
    "left_swing_interval",
    "right_swing_interval",
    "left_swing_percent",
    "right_swing_percent",
    "left_stance_interval",
    "right_stance_interval",
    "left_stance_percent",
    "right_stance_percent",
    "double_support_interval",
    "step_time",
]

control → 0  (healthy, normal gait)
als     → 1  (sick, abnormal gait)
park    → 1  (sick, abnormal gait)
hunt    → 1  (sick, abnormal gait)

In [12]:
import glob
import os

DATA_DIR = "gait-in-neurodegenerative-disease-database-1.0.0"

records = []

patterns = {
    "control": (os.path.join(DATA_DIR, "control*.ts"), 0),
    "als":     (os.path.join(DATA_DIR, "als*.ts"),     1),
    "park":    (os.path.join(DATA_DIR, "park*.ts"),    1),
    "hunt":    (os.path.join(DATA_DIR, "hunt*.ts"),    1),
}

def load_subject(filepath):
    df = pd.read_csv(filepath, sep="\t", header=None, names=COLUMNS)
    df = df[(df["left_stride_interval"] < 3) & (df["right_stride_interval"] < 3)]
    df = df[(df > 0).all(axis=1)]
    return df

for group, (pattern, label) in patterns.items():
    files = glob.glob(pattern)
    for filepath in files:
        subject_id = os.path.basename(filepath).replace(".ts", "")
        df = load_subject(filepath)
        print(f"{subject_id}: {len(df)} rows loaded")
        records.append({"id": subject_id, "group": group, "label": label, "data": df})

control1: 259 rows loaded
control10: 277 rows loaded
control11: 269 rows loaded
control12: 244 rows loaded
control13: 251 rows loaded
control14: 249 rows loaded
control15: 198 rows loaded
control16: 250 rows loaded
control2: 241 rows loaded
control3: 255 rows loaded
control4: 267 rows loaded
control5: 250 rows loaded
control6: 270 rows loaded
control7: 260 rows loaded
control8: 261 rows loaded
control9: 275 rows loaded
als1: 193 rows loaded
als10: 246 rows loaded
als11: 229 rows loaded
als12: 117 rows loaded
als13: 183 rows loaded
als2: 242 rows loaded
als3: 209 rows loaded
als4: 129 rows loaded
als5: 204 rows loaded
als6: 176 rows loaded
als7: 159 rows loaded
als8: 232 rows loaded
als9: 212 rows loaded
park1: 245 rows loaded
park10: 288 rows loaded
park11: 222 rows loaded
park12: 247 rows loaded
park13: 251 rows loaded
park14: 276 rows loaded
park15: 237 rows loaded
park2: 277 rows loaded
park3: 230 rows loaded
park4: 222 rows loaded
park5: 263 rows loaded
park6: 269 rows loaded
park7

In [13]:
def extract_features(df):
    features = {}
    cols = [c for c in COLUMNS if c != "elapsed_time"]
    for col in cols:
        s = df[col].dropna()
        features[f"{col}_mean"]   = s.mean()
        features[f"{col}_std"]    = s.std()
        features[f"{col}_cv"]     = s.std() / s.mean()   # most discriminative!
        features[f"{col}_median"] = s.median()
        features[f"{col}_iqr"]    = s.quantile(0.75) - s.quantile(0.25)
    return pd.Series(features)

In [14]:
all_features = []

for group, (pattern, label) in patterns.items():
    for filepath in glob.glob(pattern):
        subject_id = os.path.basename(filepath).replace(".ts", "")
        df = load_subject(filepath)
        feats = extract_features(df)
        feats["subject_id"] = subject_id
        feats["group"]      = group
        feats["label"]      = label
        all_features.append(feats)

dataset = pd.DataFrame(all_features)
print(dataset.shape)  # should be (64, ~65 columns)

(64, 63)


In [11]:
DATA_DIR = "gait-in-neurodegenerative-disease-database-1.0.0"

# Test it first
print(glob.glob(f"{DATA_DIR}/control*.ts"))

['gait-in-neurodegenerative-disease-database-1.0.0\\control1.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control10.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control11.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control12.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control13.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control14.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control15.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control16.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control2.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control3.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control4.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control5.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control6.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control7.ts', 'gait-in-neurodegenerative-disease-database-1.0.0\\control8.ts', 'gait-in-neurodeg

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

feature_cols = [c for c in dataset.columns 
                if c not in ("subject_id", "group", "label")]

X = dataset[feature_cols].values
y = dataset["label"].values

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

roc_scores    = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc")
recall_scores = cross_val_score(pipeline, X, y, cv=cv, scoring="recall")

print(f"ROC-AUC: {roc_scores.mean():.3f} ± {roc_scores.std():.3f}")
print(f"Recall:  {recall_scores.mean():.3f} ± {recall_scores.std():.3f}")

ROC-AUC: 0.918 ± 0.117
Recall:  0.920 ± 0.098


In [18]:
import os
print(os.getcwd()) 

C:\Users\yosrl\OneDrive\Desktop\esteps


In [19]:
import joblib
import os

pipeline.fit(X, y)
joblib.dump({"pipeline": pipeline, "features": feature_cols}, "model.pkl")

# confirm it was created
print(os.path.exists("model.pkl"))  # should print True
print(os.path.abspath("model.pkl")) # shows exact location

True
C:\Users\yosrl\OneDrive\Desktop\esteps\model.pkl
